[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qualia906/Developing-Agentic-AI-with-LangChain/blob/main/chap04/hands-on/chap04_hands_on.ipynb)


# 第4章 ハンズオン：MCPの連携とHuman-in-the-Loop (HITL)

## 学習目標
- MCP（Model Context Protocol）の概念を理解する
- `HumanInTheLoopMiddleware` を使って、重要な操作の前に人間の承認を必要とするフローを実装する
- エージェントの一時停止（interrupt）と再開（resume）を制御する

---

In [ ]:
%pip install -qU langchain langchain-openai langgraph

In [ ]:
import os
from google.colab import userdata

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "chap04-handson"
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("環境変数の設定が完了しました")

## Part 1: MCP の概念（説明）

**MCP (Model Context Protocol)** とは、LLM と外部ツール/サービスが標準化された方法で通信するためのプロトコルです。

```
【MCP の仕組み】
LLM ←→ MCP Host ←→ MCP Server ←→ 外部サービス（DB、ファイル、APIなど）
```

- **MCP Server**: ファイルシステム、データベースなどを MCP ツールとして公開するサーバー
- **MCP Host**: LangChain などのフレームワークが MCP クライアントとして機能する

このハンズオンでは、MCP の「破壊的な操作を行うツール」を模したダミーツールを使い、
それを実行する前に **人間の承認を求める HITL** の実装方法を学びます。

## Part 2: 破壊的ツールの定義（承認が必要なツール）

In [ ]:
from langchain.tools import tool

# ツール1: メール送信（承認が必要な破壊的操作の例）
@tool
def send_email(to: str, subject: str, body: str) -> str:
    """指定したアドレスにメールを送信します。
    この操作は実際の外部システム（メールサーバー）に影響を与えます。
    
    Args:
        to: 送信先のメールアドレス
        subject: メールの件名
        body: メールの本文
    """
    # 学習用ダミー実装（実際には送信しない）
    print(f"[メール送信] 宛先: {to}, 件名: {subject}")
    return f"メールを送信しました: {to} 宛, 件名: '{subject}'"


# ツール2: データ削除（承認が必要な破壊的操作の例）
@tool
def delete_record(record_id: str, table: str) -> str:
    """指定したテーブルからレコードを削除します。
    この操作は取り消せないため、十分な確認が必要です。
    
    Args:
        record_id: 削除するレコードの ID
        table: 対象のテーブル名
    """
    # 学習用ダミー実装
    print(f"[データ削除] テーブル: {table}, レコードID: {record_id}")
    return f"レコード {record_id} を {table} テーブルから削除しました"


# ツール3: 情報取得（承認不要の読み取り操作）
@tool
def get_user_info(user_id: str) -> str:
    """ユーザー情報を取得します（読み取り専用、承認不要）。
    
    Args:
        user_id: 情報を取得するユーザーの ID
    """
    return f"ユーザー情報: ID={user_id}, 名前=山田太郎, メール=yamada@example.com"


print("ツールの定義が完了しました")
print(f"承認が必要なツール: send_email, delete_record")
print(f"承認不要のツール: get_user_info")

## Part 3: HumanInTheLoopMiddleware の設定

`HumanInTheLoopMiddleware` を使うと、指定したツールの実行前に
自動的に割り込み（interrupt）が発生し、人間の確認を待つようになります。

In [ ]:
from langchain.agents import create_agent
from langchain.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# HumanInTheLoopMiddleware: 指定ツールの実行前に人間の承認を求める
hitl_middleware = HumanInTheLoopMiddleware(
    # interrupt_on: 承認が必要なツール名のセット
    interrupt_on={"send_email", "delete_record"},
)

# Checkpointer は HITL に必須（中断状態を保存するために必要）
memory = InMemorySaver()

# HITL エージェントの作成
hitl_agent = create_agent(
    model="openai:gpt-4o",
    tools=[send_email, delete_record, get_user_info],
    middleware=[hitl_middleware],   # ← middleware として HITL を追加
    checkpointer=memory,           # ← 中断状態保存のため必須
    system_prompt=(
        "あなたはビジネスオペレーション AIエージェントです。\n"
        "メール送信やデータ削除などの重要な操作は、人間の承認を得てから実行してください。\n"
        "情報取得は承認なしで実行できます。\n"
        "日本語で回答してください。"
    ),
)

print("HITL エージェントの作成が完了しました")

## Part 4: Interrupt の発生を確認する

In [ ]:
# thread_id: 会話セッションの識別子（HITL で再開するために必要）
config = {"configurable": {"thread_id": "hitl_session_001"}}

# エージェントにメール送信を依頼する
print("=== エージェントにメール送信を依頼 ===")


In [ ]:
result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "user IDが U123 のユーザー情報を取得し、そのメールアドレスに『ご利用ありがとうございます』という件名で挨拶のメールを送ってください。"
            }
        ]
    },
    config=config,
    # version="v2" を指定すると interrupt 情報が result に含まれる
    version="v2",
)

# ─ 結果の確認 ─
print()
print(f"実行結果の型: {type(result)}")

# HITL Middleware が発動すると、result に interrupt 情報が含まれる
if hasattr(result, 'interrupts') and result.interrupts:
    print()
    print("⚠️ エージェントが一時停止しました（承認待ち）")
    for interrupt in result.interrupts:
        print(f"  承認待ちのツール: {interrupt['tool_call']['name']}")
        print(f"  引数: {interrupt['tool_call']['args']}")
        print(f"  メッセージ: {interrupt.get('message', 'ご確認ください')}")
else:
    # interrupt なしで完了した場合
    if hasattr(result, 'messages'):
        print("最終回答:", result.messages[-1].content)
    else:
        print(result)

## Part 5: 承認して再開する（Resume）

In [ ]:
from langgraph.types import Command

# ─ 人間がここで判断する ─
user_decision = input("メール送信を承認しますか？ (approve/reject): ").strip().lower()

if user_decision == "approve":
    print("\n承認します。エージェントの処理を再開します...")
    
    # Command(resume=...) でエージェントを再開する
    # decisions: 各 interrupt に対して approve または reject を指定


In [ ]:
    resume_result = hitl_agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,   # 同じ thread_id で再開するために config が必要
        version="v2",
    )
    
    print()
    print("=== 承認後の実行結果 ===")
    if hasattr(resume_result, 'messages'):
        print(resume_result.messages[-1].content)
    else:
        print(resume_result)

elif user_decision == "reject":
    print("\n拒否します。エージェントの処理をキャンセルします...")
    
    # reject を指定して再開 → ツールは実行されずエージェントに通知される
    reject_result = hitl_agent.invoke(
        Command(resume={"decisions": [{"type": "reject", "reason": "ユーザーがキャンセルしました"}]}),
        config=config,
        version="v2",
    )
    
    print()
    print("=== 拒否後の実行結果 ===")
    if hasattr(reject_result, 'messages'):
        print(reject_result.messages[-1].content)
    else:
        print(reject_result)

else:
    print("approve または reject を入力してください")

## 【オプション】対話的な実行

`input()` 関数を使って、Google Colab 上で直接プロンプトを入力してエージェントを実行してみましょう。

In [ ]:
# Pythonの input() を使って対話的に質問を入力します
# 実行すると入力欄が表示されるので、質問を入力してEnterを押してください
interactive_input = input("エージェントへの入力: ")

In [ ]:
# 新たなスレッドで実行します
interactive_config = {"configurable": {"thread_id": "hitl_interactive"}}


In [ ]:
from langchain_core.messages import HumanMessage

response_interactive = agent.invoke(
    {"messages": [HumanMessage(content=interactive_input)]},
    config=interactive_config
)
print("回答:")
print(response_interactive["messages"][-1].content)
# （もし中断が発生した場合は、前のセルと同じように resume してください）

---

## まとめ

| 概念 | 実装方法 | ポイント |
|------|---------|--------|
| HITL Middleware | `HumanInTheLoopMiddleware(interrupt_on={"tool_name"})` | 指定ツールの実行前に自動で中断する |
| 中断の検出 | `result.interrupts` | 承認待ちのツール情報が含まれる |
| 承認して再開 | `Command(resume={"decisions": [{"type": "approve"}]})` | 同じ config (thread_id) を使って再開する |
| 拒否してキャンセル | `Command(resume={"decisions": [{"type": "reject"}]})` | ツールは実行されず、エージェントに通知される |

**なぜ HITL が重要なのか？**  
自律性が高まるエージェントは、誤って重要なデータを削除したり、
意図しない通知を送る可能性があります。
HITL は「エージェントの自律性」と「人間の安全な制御」のバランスを取る重要なパターンです。

**次の章（第5章）では、** 複数のエージェントを組み合わせた
Supervisor パターンのマルチエージェントシステムを構築します。